# Japan road traffic: a forecasting starter, with a road graph

Two forecasting tasks on this dataset, each with honest baselines you can try to beat:

1. **Tomorrow's traffic at each counter** from `daily_summary.csv` (one row per counter and day).
2. **The next hour**, and whether the counters **next to it on the road** (`road_graph_2021.csv`) help.

Every comparison uses a time split: the model learns from earlier days and is scored on the last three weeks it never saw.

In [ ]:
import glob, os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from sklearn.ensemble import HistGradientBoostingRegressor

# On Kaggle the files sit somewhere under /kaggle/input; elsewhere, set DATA to the folder that holds them.
hits = glob.glob("/kaggle/input/**/daily_summary.csv", recursive=True)
DATA = os.path.dirname(hits[0]) if hits else os.environ.get("DATA", ".")
plt.rcParams.update({"axes.titlesize": 16, "axes.labelsize": 14, "xtick.labelsize": 12, "ytick.labelsize": 12,
                     "legend.fontsize": 12, "axes.spines.top": False, "axes.spines.right": False})
daily = pd.read_csv(f"{DATA}/daily_summary.csv", dtype={"customary_break": "string"}, parse_dates=["date"])
print(f"{len(daily):,} counter-days, {daily.date.min():%Y-%m-%d} to {daily.date.max():%Y-%m-%d}; "
      f"{daily.vehicles.notna().mean():.0%} have all 24 hours")
daily.head()

## Holidays move traffic more than anything else

`vs_baseline` compares a day with the usual count of the same counter on the same weekday (earlier weeks only).
The median over all counters stays near 1 on ordinary days and jumps in runs of days off.

In [ ]:
full = daily[daily.vehicles.notna() & daily.vs_baseline.notna()]
nat = full.groupby("date").agg(ratio=("vs_baseline", "median"), run=("days_off_in_run", "first"),
                                hol=("is_holiday", "first"), brk=("customary_break", "first"))
fig, ax = plt.subplots(figsize=(13, 4.5))
for d, r in nat[nat.hol | (nat.run >= 3) | nat.brk.notna()].iterrows():
    ax.axvspan(d - pd.Timedelta(hours=12), d + pd.Timedelta(hours=12), color="#f2c14e", alpha=0.35, lw=0)
ax.plot(nat.index, nat.ratio, color="#3b6ea5", lw=2)
ax.axhline(1, color="grey", lw=1)
ax.set_title("Traffic against the usual day rises around holidays (shaded: holidays, 3+ days off, Obon)")
ax.set_ylabel("median vs_baseline")
plt.tight_layout(); plt.show()

The highest day of the summer is Monday 2026-08-10, a working day between a weekend and Mountain Day (Tuesday 08-11),
in the run-up to Obon. The dips need reading with the definition in mind: the usual count counts holidays as Sundays
and takes the 4 most recent such days, so the last day of the five-day September break (09-23) and the two Sundays
after it (09-27 and 10-04) are compared with the busy days of the break itself and look quiet.

## Task 1: tomorrow's vehicles at each counter

Predict a counter's vehicles for a day from what is known the evening before: its own last days, the usual count for that
weekday (`baseline_vehicles` uses earlier days only), and the calendar. Scored by WAPE (total absolute error / total
vehicles) on the last 21 days.

In [ ]:
d = full[full.vehicles > 0].copy()
d["y"] = np.log(d.vehicles)
key = ["counter_id", "sensor"]
idx = d.set_index(key + ["date"])

def shifted(col, days):
    s = idx[col].copy()
    s.index = s.index.set_levels(s.index.levels[2] + pd.Timedelta(days=days), level=2)
    return s.reindex(idx.index).to_numpy()

d = d.sort_values(key + ["date"]).reset_index(drop=True)
idx = d.set_index(key + ["date"])
d["lag1"], d["lag7"], d["lag14"] = shifted("y", 1), shifted("y", 7), shifted("y", 14)
d["vsb1"] = np.log(shifted("vs_baseline", 1).astype(float))
d["dayoff1"] = shifted("day_off", 1).astype(float)
d["base"] = np.log(d.baseline_vehicles.astype(float))
for c in ("day_off", "is_holiday"):
    d[c] = d[c].astype(float)
FEATS = ["lag1", "lag7", "lag14", "vsb1", "base", "weekday", "day_off", "days_off_in_run", "is_holiday", "dayoff1"]
cut = d.date.max() - pd.Timedelta(days=20)
train, test = d[d.date < cut], d[(d.date >= cut) & d[["lag1", "lag7", "base"]].notna().all(axis=1)].copy()
model = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=0)
model.fit(train[train.lag1.notna()][FEATS], train[train.lag1.notna()].y)
test["model"] = model.predict(test[FEATS])

def wape(t, col):
    return np.abs(t.vehicles - np.exp(t[col])).sum() / t.vehicles.sum()

rows = []
for name, sel in (("all days", test.index == test.index), ("working days", test.days_off_in_run == 0),
                  ("ordinary weekends (2 days off)", test.days_off_in_run == 2),
                  ("runs of 3+ days off", test.days_off_in_run >= 3)):
    t = test[sel]
    rows.append({"days": name, "n": len(t), "same weekday last week": wape(t, "lag7"),
                 "usual count (baseline)": wape(t, "base"), "gradient boosting": wape(t, "model")})
scores = pd.DataFrame(rows).set_index("days")
print(f"test {test.date.min():%Y-%m-%d} to {test.date.max():%Y-%m-%d}")
scores.style.format({c: "{:.1%}" for c in scores.columns if c != "n"})

In [ ]:
ax = scores.drop(columns="n").plot.barh(figsize=(11, 5), color=["#bbbbbb", "#7aa974", "#3b6ea5"], width=0.75)
ax.set_title("Days off are where a model beats the usual count")
ax.set_ylabel("")
ax.legend(title=None, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
ax.set_xlabel("WAPE on the last 21 days (lower is better)")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

On ordinary working days the usual count is already about as good as the model; the gain comes on days off, most of all
on ordinary weekends. In the October 2026 test window the usual count of a weekend still holds days of the September
break (its Saturday 09-19, and its holidays, counted as Sundays), while the calendar columns tell the model what kind of
day each one was. **Beat the gradient-boosting line**: it is a plain model with ten features.

## Task 2: the next hour, with the road graph

`road_graph_2021.csv` gives the road distance between 365 counters tied to the 2021 Road Traffic Census; `direct` marks
pairs with no other counter in between. Here a model predicts each counter's next-hour departure from its usual count
(log `vs_baseline`) from its own last three hours, with and without the **mean of its direct road neighbours** at the
current hour. As a check, the same feature is built from the neighbours of a **random other counter** (several draws).

In [ ]:
graph = pd.read_csv(f"{DATA}/road_graph_2021.csv")
direct = graph[graph.direct]
ids = sorted(set(direct.counter_a) | set(direct.counter_b))
files = sorted(glob.glob(f"{DATA}/hourly_*.parquet"))
h = ds.dataset(files).to_table(columns=["time_jst", "counter_id", "sensor", "vs_baseline", "day_off", "days_off_in_run"],
                               filter=(ds.field("sensor") == "loop") & ds.field("counter_id").isin(ids)).to_pandas()
h = h[h.vs_baseline > 0]
Z = h.assign(z=np.log(h.vs_baseline.astype(float))).pivot_table(index="time_jst", columns="counter_id", values="z")
Z = Z.reindex(pd.date_range(Z.index.min(), Z.index.max(), freq="h"))
nbrs = {c: [] for c in Z.columns}
for a, b in zip(direct.counter_a, direct.counter_b):
    if a in nbrs and b in nbrs:
        nbrs[a].append(b); nbrs[b].append(a)
cols = [c for c in Z.columns if nbrs[c]]
calh = h.drop_duplicates("time_jst").set_index("time_jst")[["day_off", "days_off_in_run"]].astype(float)
print(f"{len(cols)} counters with a direct neighbour, {len(Z):,} hours")

In [ ]:
def table(neigh):
    parts = []
    for c in cols:
        s, n = Z[c], Z[neigh[c]].mean(axis=1)
        parts.append(pd.DataFrame({"t": Z.index, "y": s.shift(-1).to_numpy(), "z0": s.to_numpy(), "z1": s.shift(1).to_numpy(),
                                   "z2": s.shift(2).to_numpy(), "nb": n.to_numpy(), "nb1": n.shift(1).to_numpy()}))
    t = pd.concat(parts, ignore_index=True)
    nxt = t.t + pd.Timedelta(hours=1)
    t["hour"] = nxt.dt.hour
    t = t.join(calh.reindex(nxt).reset_index(drop=True))
    return t[t[["y", "z0", "z1", "z2", "nb"]].notna().all(axis=1)]

def score(t, feats):
    cut = t.t.max() - pd.Timedelta(days=21)
    tr, te = t[t.t < cut], t[t.t >= cut]
    m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0).fit(tr[feats], tr.y)
    return np.abs(te.y - m.predict(te[feats])).mean()

OWN = ["z0", "z1", "z2", "hour", "day_off", "days_off_in_run"]
real = table(nbrs)
res = {"own hours only": score(real, OWN), "+ road neighbours": score(real, OWN + ["nb", "nb1"])}
rng = np.random.default_rng(0)
placebo = []
for k in range(5):
    perm = dict(zip(cols, rng.permutation(cols)))
    placebo.append(score(table({c: nbrs[perm[c]] for c in cols}), OWN + ["nb", "nb1"]))
res["+ random counter's neighbours (5 draws, mean)"] = float(np.mean(placebo))
print({k: round(v, 5) for k, v in res.items()}, "placebo draws", np.round(placebo, 5))

In [ ]:
own = res["own hours only"]
gain = {k: 1 - v / own for k, v in res.items() if k != "own hours only"}
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.barh(list(gain), [100 * v for v in gain.values()], color=["#3b6ea5", "#bbbbbb"])
ax.errorbar([100 * (1 - p / own) for p in placebo], [1] * len(placebo), fmt="o", color="#555555", ms=5)
ax.axvline(0, color="grey", lw=1)
ax.set_title("One hour ahead, the counters next to it on the road help")
ax.set_xlabel("error reduction against own hours only, %")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

The road neighbours cut the next-hour error a little; a random counter's neighbours, which carry the same nationwide
swings but not the road, cut it less (dots: each draw). The day-ahead model in Task 1 gets nothing from yesterday's
neighbours (not shown; tried with the same placebo), so the road graph is a short-horizon signal.

## Where to go from here
- `five_minute_kanto_*.parquet` has the Kanto box every five minutes: the horizon at which road neighbours should matter
  most, and the setting of graph models such as DCRNN or Graph WaveNet (a PEMS-BAY-style problem with Japanese roads).
- `road_km` gives a distance for a Gaussian-kernel adjacency; `direct` gives a sparse one.
- Rain (`precip_mm`, `wet_hours`) and the calendar are ready to use as features in both files.
- Counts, not speeds: these files say how many vehicles passed, not how fast.